# Online Conv2D Autoencoder with Continual Learning| Strategy | Buffer Size | Replay Weight | Replay Batch ||----------|------------|---------------|-------------|| **Naive** (no CL) | - | - | - || **ER Scaled** | 100 | 0.7 | 30 || **ER Aggressive** | 200 | 1.0 | 60 |**Architecture**: Conv2D AE processing per-timestep grid images (4 x 32 x 128)\**Training**: 20 temporal windows x 15 timesteps/window x 100 epochs/window\**Note**: Buffer sizes are smaller than INR/Linear AE because each Conv2D sample is a full grid image (16K floats vs 4-60 floats per INR/LAE sample)

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
"""
Online Conv2D Autoencoder with Continual Learning Strategies
"""

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import pyarrow.csv as pv
from scipy.spatial import Delaunay, cKDTree
from scipy.interpolate import LinearNDInterpolator, RegularGridInterpolator
from torcheval.metrics import PeakSignalNoiseRatio
from torchmetrics.image import StructuralSimilarityIndexMeasure
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
import time
import os
import json


# Dataset: Grid images with windowing support
class GridImageDataset(Dataset):
    """
    Converts unstructured mesh data to regular grid images for Conv2D.
    Each sample: one timestep as (4, grid_h, grid_w).
    Supports temporal windowing for online training.
    """

    def __init__(self, filepath, grid_h=32, grid_w=128):
        print("Loading dataset from: {}".format(filepath))
        read_options = pv.ReadOptions(
            column_names=['x', 'y', 'z', 't', 'Vx', 'Vy', 'Pressure', 'TKE']
        )
        table = pv.read_csv(filepath, read_options=read_options)
        data = table.to_pandas()
        data = data.sort_values(['x', 'y', 'z', 't']).reset_index(drop=True)

        self.time_values = np.sort(data['t'].unique()).astype(np.float32)
        self.num_timesteps = len(self.time_values)
        self.num_vars = 4
        self.var_names = ['Vx', 'Vy', 'Pressure', 'TKE']
        self.grid_h = grid_h
        self.grid_w = grid_w

        fields = data[['Vx', 'Vy', 'Pressure', 'TKE']].values.astype(np.float32)
        self.num_points = len(data) // self.num_timesteps
        self.mesh_coords = data[['x', 'y']].values[::self.num_timesteps].astype(np.float32)

        # Global min-max normalization
        self.field_min = fields.min(axis=0)
        self.field_max = fields.max(axis=0)
        self.field_range = self.field_max - self.field_min
        self.field_range[self.field_range == 0] = 1.0

        fields_3d = fields.reshape(self.num_points, self.num_timesteps, self.num_vars)
        fields_3d = fields_3d.transpose(1, 0, 2)  # (T, N, V)
        self.mesh_fields_norm = (fields_3d - self.field_min) / self.field_range

        print("Detected {} spatial points across {} timesteps".format(
            self.num_points, self.num_timesteps))

        # Regular grid
        x_min, x_max = self.mesh_coords[:, 0].min(), self.mesh_coords[:, 0].max()
        y_min, y_max = self.mesh_coords[:, 1].min(), self.mesh_coords[:, 1].max()
        self.grid_x = np.linspace(x_min, x_max, grid_w).astype(np.float32)
        self.grid_y = np.linspace(y_min, y_max, grid_h).astype(np.float32)
        self.gx, self.gy = np.meshgrid(self.grid_x, self.grid_y)

        print("Interpolating to regular grid ({} x {})...".format(grid_h, grid_w))
        tri = Delaunay(self.mesh_coords)

        # NaN mask for boundary points
        dummy = LinearNDInterpolator(tri, np.ones(self.num_points))
        dummy_vals = dummy(self.gx, self.gy)
        self.nan_mask = np.isnan(dummy_vals)

        if self.nan_mask.any():
            nan_xy = np.column_stack([self.gx[self.nan_mask], self.gy[self.nan_mask]])
            tree = cKDTree(self.mesh_coords)
            _, self.nearest_idx = tree.query(nan_xy)
            print("  {} grid points filled via nearest neighbor".format(self.nan_mask.sum()))

        # Interpolate all timesteps
        grid_data = np.zeros((self.num_timesteps, self.num_vars, grid_h, grid_w), dtype=np.float32)
        for t in range(self.num_timesteps):
            for v in range(self.num_vars):
                values = self.mesh_fields_norm[t, :, v]
                lin = LinearNDInterpolator(tri, values)
                gv = lin(self.gx, self.gy)
                if self.nan_mask.any():
                    gv[self.nan_mask] = values[self.nearest_idx]
                grid_data[t, v] = gv
            if (t + 1) % 50 == 0:
                print("  Interpolated {}/{} timesteps".format(t + 1, self.num_timesteps))

        self.grid_data = torch.FloatTensor(grid_data)
        print("Grid dataset ready: {} samples of shape ({}, {}, {})".format(
            self.num_timesteps, self.num_vars, grid_h, grid_w))

    def get_window_data(self, window_idx, num_windows=20):
        """Get grid images for a specific temporal window."""
        time_seq = self.num_timesteps // num_windows
        start_t = window_idx * time_seq
        end_t = start_t + time_seq if window_idx < num_windows - 1 else self.num_timesteps
        return self.grid_data[start_t:end_t]  # (time_seq, 4, H, W)

    def grid_to_mesh(self, grid_predictions):
        """Interpolate grid predictions back to original mesh points."""
        single = grid_predictions.ndim == 3
        if single:
            grid_predictions = grid_predictions[np.newaxis]
        T = grid_predictions.shape[0]
        mesh_preds = np.zeros((T, self.num_points, self.num_vars), dtype=np.float32)
        query_pts = np.column_stack([self.mesh_coords[:, 1], self.mesh_coords[:, 0]])
        for t in range(T):
            for v in range(self.num_vars):
                interp = RegularGridInterpolator(
                    (self.grid_y, self.grid_x), grid_predictions[t, v],
                    method='linear', bounds_error=False, fill_value=None)
                mesh_preds[t, :, v] = interp(query_pts)
        return mesh_preds[0] if single else mesh_preds

    def __len__(self):
        return self.num_timesteps

    def __getitem__(self, idx):
        x = self.grid_data[idx]
        return x, x

    def get_normalization_params(self):
        return {
            'field_min': self.field_min.tolist(),
            'field_max': self.field_max.tolist(),
            'field_range': self.field_range.tolist(),
            'num_timesteps': self.num_timesteps,
            'num_points': self.num_points,
            'num_vars': self.num_vars,
            'grid_h': self.grid_h,
            'grid_w': self.grid_w,
        }


# Models
class ConvEncoder(nn.Module):
    """Conv2D encoder with stride-2 downsampling and linear bottleneck."""
    def __init__(self, in_channels, channel_list, latent_dim):
        super().__init__()
        conv_layers = []
        prev_ch = in_channels
        for ch in channel_list:
            conv_layers.extend([
                nn.Conv2d(prev_ch, ch, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(ch),
                nn.LeakyReLU(0.1),
            ])
            prev_ch = ch
        self.conv = nn.Sequential(*conv_layers)
        self.latent_dim = latent_dim
        self._flatten_dim = None
        self._fc = None

    def _build_fc(self, x):
        self._flatten_dim = x.shape[1] * x.shape[2] * x.shape[3]
        self._fc = nn.Linear(self._flatten_dim, self.latent_dim).to(x.device)

    def forward(self, x):
        x = self.conv(x)
        if self._fc is None:
            self._build_fc(x)
        x = x.view(x.size(0), -1)
        return self._fc(x)


class ConvDecoder(nn.Module):
    """Conv2D decoder with stride-2 upsampling from linear bottleneck."""
    def __init__(self, out_channels, channel_list, latent_dim, bottleneck_shape):
        super().__init__()
        self.bottleneck_shape = bottleneck_shape
        flatten_dim = bottleneck_shape[0] * bottleneck_shape[1] * bottleneck_shape[2]
        self.fc = nn.Linear(latent_dim, flatten_dim)
        conv_layers = []
        reversed_ch = list(reversed(channel_list))
        for i in range(len(reversed_ch) - 1):
            conv_layers.extend([
                nn.ConvTranspose2d(reversed_ch[i], reversed_ch[i + 1],
                                   kernel_size=3, stride=2, padding=1, output_padding=1),
                nn.BatchNorm2d(reversed_ch[i + 1]),
                nn.LeakyReLU(0.1),
            ])
        conv_layers.append(
            nn.ConvTranspose2d(reversed_ch[-1], out_channels,
                               kernel_size=3, stride=2, padding=1, output_padding=1))
        self.conv = nn.Sequential(*conv_layers)

    def forward(self, z):
        x = self.fc(z)
        x = x.view(x.size(0), *self.bottleneck_shape)
        return self.conv(x)


class ConvAutoEncoder(nn.Module):
    """Conv2D autoencoder for per-timestep grid data."""
    def __init__(self, in_channels, channel_list, latent_dim, input_shape):
        super().__init__()
        self.latent_dim = latent_dim
        # Compute bottleneck shape
        dummy = torch.zeros(1, in_channels, *input_shape)
        temp_conv = nn.Sequential(*[
            nn.Conv2d(c_in, c_out, kernel_size=3, stride=2, padding=1)
            for c_in, c_out in zip([in_channels] + channel_list[:-1], channel_list)
        ])
        with torch.no_grad():
            dummy_out = temp_conv(dummy)
        bottleneck_shape = (channel_list[-1], dummy_out.shape[2], dummy_out.shape[3])
        self.encoder = ConvEncoder(in_channels, channel_list, latent_dim)
        self.decoder = ConvDecoder(in_channels, channel_list, latent_dim, bottleneck_shape)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)


class AEForwardWrapper(nn.Module):
    """Wraps AE so forward() returns only reconstruction (strategy compatibility)."""
    def __init__(self, ae):
        super().__init__()
        self.ae = ae
    def forward(self, x):
        x_hat, _ = self.ae(x)
        return x_hat


CONV_AE_CONFIGS = {
    'base': {'channel_list': [16, 32, 64, 128], 'latent_dim': 32},
    'medium': {'channel_list': [32, 64, 128, 256], 'latent_dim': 64},
    'large': {'channel_list': [32, 64, 128, 256], 'latent_dim': 128},
}


def create_conv_ae(size, input_shape, in_channels=4):
    cfg = CONV_AE_CONFIGS[size]
    return ConvAutoEncoder(in_channels, cfg['channel_list'], cfg['latent_dim'], input_shape)


# CL Strategies
class ReplayBuffer:
    """Reservoir sampling buffer: works with any tensor shape."""
    def __init__(self, max_size=100):
        self.max_size = max_size
        self.inputs = None
        self.targets = None
        self.count = 0
        self.current_size = 0

    def add_window_batch(self, inputs, targets, n_samples=None):
        window_size = inputs.shape[0]
        if n_samples is None:
            n_samples = min(window_size, self.max_size // 2)
        indices = torch.randperm(window_size)[:n_samples]
        self._add(inputs[indices], targets[indices])

    def _add(self, inputs, targets):
        inp_cpu = inputs.detach().cpu()
        tgt_cpu = targets.detach().cpu()
        n = inp_cpu.shape[0]
        if self.inputs is None:
            n_init = min(n, self.max_size)
            self.inputs = inp_cpu[:n_init].clone()
            self.targets = tgt_cpu[:n_init].clone()
            self.current_size = n_init
            self.count = n_init
            start = n_init
        else:
            start = 0
        for i in range(start, n):
            self.count += 1
            if self.current_size < self.max_size:
                self.inputs = torch.cat([self.inputs, inp_cpu[i:i+1]], dim=0)
                self.targets = torch.cat([self.targets, tgt_cpu[i:i+1]], dim=0)
                self.current_size += 1
            else:
                j = np.random.randint(0, self.count)
                if j < self.max_size:
                    self.inputs[j] = inp_cpu[i]
                    self.targets[j] = tgt_cpu[i]

    def sample(self, batch_size, device=None):
        actual = min(batch_size, self.current_size)
        idx = torch.randperm(self.current_size)[:actual]
        bi, bt = self.inputs[idx], self.targets[idx]
        if device:
            bi, bt = bi.to(device), bt.to(device)
        return bi, bt

    def __len__(self):
        return self.current_size


class NaiveStrategy:
    def __init__(self):
        self.name = "naive"
    def before_window(self, model, window_idx, wi, wt, device): pass
    def compute_loss(self, model, criterion, outputs, targets, window_inputs, device):
        return criterion(outputs, targets)
    def after_window(self, model, window_idx, wi, wt, device): pass
    def get_config(self):
        return {"strategy": self.name}


class ERStrategy:
    def __init__(self, name, buffer_size, replay_weight, replay_batch_size):
        self.name = name
        self.buffer = ReplayBuffer(max_size=buffer_size)
        self.replay_weight = replay_weight
        self.replay_batch_size = replay_batch_size

    def before_window(self, model, window_idx, wi, wt, device): pass

    def compute_loss(self, model, criterion, outputs, targets, window_inputs, device):
        current_loss = criterion(outputs, targets)
        if len(self.buffer) == 0:
            return current_loss
        ri, rt = self.buffer.sample(self.replay_batch_size, device=device)
        replay_outputs = model(ri)
        replay_loss = criterion(replay_outputs, rt)
        return current_loss + self.replay_weight * replay_loss

    def after_window(self, model, window_idx, wi, wt, device):
        self.buffer.add_window_batch(wi, wt)

    def get_config(self):
        return {"strategy": self.name, "buffer_size": self.buffer.max_size,
                "replay_weight": self.replay_weight, "replay_batch_size": self.replay_batch_size}


# Metrics
def compute_psnr_ssim(predictions, targets, device):
    predictions, targets = predictions.to(device), targets.to(device)
    psnr_metric = PeakSignalNoiseRatio().to(device)
    psnr_metric.update(predictions, targets)
    psnr = psnr_metric.compute().item()
    pred_ssim = predictions.unsqueeze(0).unsqueeze(0)
    target_ssim = targets.unsqueeze(0).unsqueeze(0)
    ssim_metric = StructuralSimilarityIndexMeasure(gaussian_kernel=False, kernel_size=1).to(device)
    ssim_metric.update(pred_ssim, target_ssim)
    ssim = ssim_metric.compute().item()
    return psnr, ssim


def compute_relative_error(predictions, targets):
    return (torch.norm(predictions - targets) / torch.norm(targets) * 100).item()


# Online Training Loop
def train_online_conv_cl(model, dataset, device, epochs_per_window, model_name,
                         output_dir, strategy, num_windows=20):
    os.makedirs(output_dir, exist_ok=True)
    wrapped = AEForwardWrapper(model).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    time_seq = dataset.num_timesteps // num_windows

    metrics = {'window': [], 'loss': [], 'psnr': [], 'ssim': [],
               'relative_error': [], 'time_per_window': []}
    total_params = sum(p.numel() for p in model.parameters())

    print("\n[Training] {} | {} | {:,} params | latent={}".format(
        model_name, strategy.name, total_params, model.latent_dim))
    print("  Grid: {}x{} | {} windows x {} timesteps | {} epochs/window".format(
        dataset.grid_h, dataset.grid_w, num_windows, time_seq, epochs_per_window))

    total_start = time.time()
    for window_idx in range(num_windows):
        window_start = time.time()
        window_data = dataset.get_window_data(window_idx, num_windows).to(device)

        strategy.before_window(wrapped, window_idx, window_data, window_data, device)

        wrapped.train()
        for epoch in range(epochs_per_window):
            optimizer.zero_grad()
            outputs = wrapped(window_data)
            loss = strategy.compute_loss(wrapped, criterion, outputs,
                                          targets=window_data, window_inputs=window_data, device=device)
            loss.backward()
            optimizer.step()

        strategy.after_window(wrapped, window_idx, window_data, window_data, device)

        wrapped.eval()
        with torch.no_grad():
            preds = wrapped(window_data)
        loss_val = criterion(preds, window_data).item()

        # Flatten for metrics: (T, 4, H, W) -> (T*H*W, 4)
        p_flat = preds.permute(0, 2, 3, 1).reshape(-1, dataset.num_vars)
        t_flat = window_data.permute(0, 2, 3, 1).reshape(-1, dataset.num_vars)
        psnr, ssim = compute_psnr_ssim(p_flat, t_flat, device)
        rel_error = compute_relative_error(p_flat, t_flat)
        wt = time.time() - window_start

        metrics['window'].append(window_idx + 1)
        metrics['loss'].append(loss_val)
        metrics['psnr'].append(psnr)
        metrics['ssim'].append(ssim)
        metrics['relative_error'].append(rel_error)
        metrics['time_per_window'].append(wt)

        print("  Window {:2d}/{}: PSNR={:.2f} dB, SSIM={:.4f}, RE={:.2f}%, Time={:.2f}s".format(
            window_idx + 1, num_windows, psnr, ssim, rel_error, wt))

    total_time = time.time() - total_start
    torch.save(model.state_dict(), os.path.join(output_dir, '{}_final.pth'.format(model_name)))
    with open(os.path.join(output_dir, '{}_normalization.json'.format(model_name)), 'w') as f:
        json.dump(dataset.get_normalization_params(), f, indent=2)

    print("[Done] {:.2f}s | Final: PSNR={:.2f}, SSIM={:.4f}, RE={:.2f}%".format(
        total_time, metrics['psnr'][-1], metrics['ssim'][-1], metrics['relative_error'][-1]))
    return metrics


def evaluate_full_dataset_conv(model, dataset, device, model_name=None, num_windows=20):
    """Evaluate on all windows, optionally interpolate back to mesh for fair comparison."""
    wrapped = AEForwardWrapper(model).to(device)
    wrapped.eval()

    all_preds, all_targets = [], []
    for w in range(num_windows):
        wd = dataset.get_window_data(w, num_windows).to(device)
        with torch.no_grad():
            all_preds.append(wrapped(wd))
        all_targets.append(wd)
    all_preds = torch.cat(all_preds, dim=0)  # (300, 4, H, W)
    all_targets = torch.cat(all_targets, dim=0)

    # Grid-level metrics
    p_flat = all_preds.permute(0, 2, 3, 1).reshape(-1, dataset.num_vars)
    t_flat = all_targets.permute(0, 2, 3, 1).reshape(-1, dataset.num_vars)
    psnr_grid, ssim_grid = compute_psnr_ssim(p_flat, t_flat, device)
    re_grid = compute_relative_error(p_flat, t_flat)

    # Mesh-level metrics (fair comparison with INR and Linear AE)
    mesh_preds = dataset.grid_to_mesh(np.clip(all_preds.cpu().numpy(), 0, 1))
    mesh_pred_flat = torch.FloatTensor(mesh_preds.reshape(-1, dataset.num_vars))
    mesh_tgt_flat = torch.FloatTensor(dataset.mesh_fields_norm.reshape(-1, dataset.num_vars))
    psnr_mesh, ssim_mesh = compute_psnr_ssim(mesh_pred_flat, mesh_tgt_flat, device)
    re_mesh = compute_relative_error(mesh_pred_flat, mesh_tgt_flat)

    label = model_name or "Conv AE"
    print("[Full Eval] {} | Grid: PSNR={:.2f}, Mesh: PSNR={:.2f} dB".format(label, psnr_grid, psnr_mesh))
    return {
        'psnr_db': psnr_mesh, 'ssim': ssim_mesh, 'relative_error_pct': re_mesh,
        'psnr_grid': psnr_grid, 'ssim_grid': ssim_grid, 're_grid': re_grid,
        'training_time_s': 0,
    }


# Flow Field Visualization
def visualize_conv_flow_field(model, dataset, device, timestep_idx=0,
                               title='Conv AE Flow Field', save_path=None, num_windows=20):
    """Visualize Conv2D AE reconstruction at mesh level for a specific timestep."""
    wrapped = AEForwardWrapper(model).to(device)
    wrapped.eval()

    time_seq = dataset.num_timesteps // num_windows
    window_idx = min(timestep_idx // time_seq, num_windows - 1)
    local_idx = timestep_idx - window_idx * time_seq

    window_data = dataset.get_window_data(window_idx, num_windows).to(device)
    with torch.no_grad():
        preds_grid = wrapped(window_data).cpu().numpy()

    # Interpolate single timestep back to mesh
    pred_mesh = dataset.grid_to_mesh(np.clip(preds_grid[local_idx:local_idx+1], 0, 1))
    target_mesh = dataset.mesh_fields_norm[timestep_idx]

    pred_t = pred_mesh[0]
    abs_err = np.abs(target_mesh - pred_t)
    x = dataset.mesh_coords[:, 0]
    y = dataset.mesh_coords[:, 1]

    fig = plt.figure(figsize=(20, 20))
    gs = GridSpec(4, 5, figure=fig, width_ratios=[1, 1, 0.05, 1, 0.05],
                  wspace=0.35, hspace=0.25)

    for row in range(4):
        original = target_mesh[:, row]
        predicted = pred_t[:, row]
        error = abs_err[:, row]

        ax0 = fig.add_subplot(gs[row, 0])
        ax0.scatter(x, y, c=original, cmap='jet', s=0.5, alpha=0.8, vmin=0, vmax=1)
        ax0.set_title('Original: {}'.format(dataset.var_names[row]))
        ax0.set_aspect('equal'); ax0.grid(True, alpha=0.3)

        ax1 = fig.add_subplot(gs[row, 1])
        sc2 = ax1.scatter(x, y, c=predicted, cmap='jet', s=0.5, alpha=0.8, vmin=0, vmax=1)
        ax1.set_title('Predicted: {}'.format(dataset.var_names[row]))
        ax1.set_aspect('equal'); ax1.grid(True, alpha=0.3)

        cax1 = fig.add_subplot(gs[row, 2])
        fig.colorbar(sc2, cax=cax1)

        ax2 = fig.add_subplot(gs[row, 3])
        sc3 = ax2.scatter(x, y, c=error, cmap='hot', s=0.5, alpha=0.8, vmin=0, vmax=1)
        ax2.set_title('Abs Error: {}'.format(dataset.var_names[row]))
        ax2.set_aspect('equal'); ax2.grid(True, alpha=0.3)

        cax2 = fig.add_subplot(gs[row, 4])
        fig.colorbar(sc3, cax=cax2)

    fig.suptitle(title, fontsize=14, fontweight='bold', y=0.98)
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print("  Timestep: {:.4f} (index {}, window {})".format(
        dataset.time_values[timestep_idx], timestep_idx, window_idx + 1))


print("All utilities loaded.")

In [ ]:
DATA_FILE = "/kaggle/input/ml-test-loader-original-data-csv/ML_test_loader_original_data.csv"
EPOCHS_PER_WINDOW = 100
NUM_WINDOWS = 20
GRID_H = 32
GRID_W = 128
VIS_TIMESTEP = 150

STRATEGIES = {
    'naive': lambda: NaiveStrategy(),
    'er_scaled': lambda: ERStrategy('er_scaled', 100, 0.7, 30),
    'er_aggressive': lambda: ERStrategy('er_aggressive', 200, 1.0, 60),
}

CONV_OFFLINE_REF = {
    'base':   {'psnr': 30.67, 'ssim': 0.9574, 're': 5.22},
    'medium': {'psnr': 32.75, 'ssim': 0.9723, 're': 4.10},
    'large':  {'psnr': 32.75, 'ssim': 0.9704, 're': 4.10},
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device: {}".format(device))

dataset = GridImageDataset(DATA_FILE, grid_h=GRID_H, grid_w=GRID_W)
input_shape = (GRID_H, GRID_W)

all_metrics = {}
all_evals = {}

---## 1. Naive Online (No CL)

In [ ]:
for model_name in ['base', 'medium', 'large']:
    out_dir = "/kaggle/working/results/conv_naive_{}/".format(model_name)
    model = create_conv_ae(model_name, input_shape).to(device)
    strategy = STRATEGIES['naive']()
    metrics = train_online_conv_cl(
        model=model, dataset=dataset, device=device,
        epochs_per_window=EPOCHS_PER_WINDOW, model_name=model_name,
        output_dir=out_dir, strategy=strategy, num_windows=NUM_WINDOWS)
    eval_res = evaluate_full_dataset_conv(model, dataset, device, model_name=model_name, num_windows=NUM_WINDOWS)
    eval_res['training_time_s'] = sum(metrics['time_per_window'])
    all_metrics[(model_name, 'naive')] = metrics
    all_evals[(model_name, 'naive')] = eval_res

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for mn, col in [('base', 'tab:blue'), ('medium', 'tab:orange'), ('large', 'tab:green')]:
    m = all_metrics[(mn, 'naive')]
    axes[0,0].plot(m['window'], m['loss'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
    axes[0,1].plot(m['window'], m['psnr'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
    axes[1,0].plot(m['window'], m['ssim'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
    axes[1,1].plot(m['window'], m['relative_error'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
for ax in axes.flat: ax.legend(); ax.grid(True, alpha=0.3)
axes[0,0].set(xlabel='Window', ylabel='Loss', title='Training Loss')
axes[0,1].set(xlabel='Window', ylabel='PSNR (dB)', title='PSNR')
axes[1,0].set(xlabel='Window', ylabel='SSIM', title='SSIM')
axes[1,1].set(xlabel='Window', ylabel='RE (%)', title='Relative Error')
plt.suptitle('Online Conv AE: Naive (No CL): Per-Window Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/results/conv_naive_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
best_mn = max(['base', 'medium', 'large'], key=lambda m: all_evals[(m, 'naive')]['psnr_db'])
best_psnr = all_evals[(best_mn, 'naive')]['psnr_db']
best_model = create_conv_ae(best_mn, input_shape).to(device)
with torch.no_grad(): best_model(torch.zeros(1, 4, *input_shape).to(device))  # init lazy _fc
best_model.load_state_dict(torch.load(
    '/kaggle/working/results/conv_naive_{}/{}_final.pth'.format(best_mn, best_mn), map_location=device))
print('Best Naive model: {} (PSNR: {:.2f} dB)'.format(best_mn, best_psnr))
visualize_conv_flow_field(
    best_model, dataset, device, timestep_idx=VIS_TIMESTEP,
    title='Online Conv AE Naive ({}): PSNR: {:.2f} dB'.format(best_mn, best_psnr),
    save_path='/kaggle/working/results/conv_naive_flow_field.png', num_windows=NUM_WINDOWS)

---## 2. ER Scaled

In [ ]:
for model_name in ['base', 'medium', 'large']:
    out_dir = "/kaggle/working/results/conv_er_scaled_{}/".format(model_name)
    model = create_conv_ae(model_name, input_shape).to(device)
    strategy = STRATEGIES['er_scaled']()
    metrics = train_online_conv_cl(
        model=model, dataset=dataset, device=device,
        epochs_per_window=EPOCHS_PER_WINDOW, model_name=model_name,
        output_dir=out_dir, strategy=strategy, num_windows=NUM_WINDOWS)
    eval_res = evaluate_full_dataset_conv(model, dataset, device, model_name=model_name, num_windows=NUM_WINDOWS)
    eval_res['training_time_s'] = sum(metrics['time_per_window'])
    all_metrics[(model_name, 'er_scaled')] = metrics
    all_evals[(model_name, 'er_scaled')] = eval_res

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for mn, col in [('base', 'tab:blue'), ('medium', 'tab:orange'), ('large', 'tab:green')]:
    m = all_metrics[(mn, 'er_scaled')]
    axes[0,0].plot(m['window'], m['loss'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
    axes[0,1].plot(m['window'], m['psnr'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
    axes[1,0].plot(m['window'], m['ssim'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
    axes[1,1].plot(m['window'], m['relative_error'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
for ax in axes.flat: ax.legend(); ax.grid(True, alpha=0.3)
axes[0,0].set(xlabel='Window', ylabel='Loss', title='Training Loss')
axes[0,1].set(xlabel='Window', ylabel='PSNR (dB)', title='PSNR')
axes[1,0].set(xlabel='Window', ylabel='SSIM', title='SSIM')
axes[1,1].set(xlabel='Window', ylabel='RE (%)', title='Relative Error')
plt.suptitle('Online Conv AE: ER Scaled: Per-Window Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/results/conv_er_scaled_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
best_mn = max(['base', 'medium', 'large'], key=lambda m: all_evals[(m, 'er_scaled')]['psnr_db'])
best_psnr = all_evals[(best_mn, 'er_scaled')]['psnr_db']
best_model = create_conv_ae(best_mn, input_shape).to(device)
with torch.no_grad(): best_model(torch.zeros(1, 4, *input_shape).to(device))  # init lazy _fc
best_model.load_state_dict(torch.load(
    '/kaggle/working/results/conv_er_scaled_{}/{}_final.pth'.format(best_mn, best_mn), map_location=device))
print('Best ER Scaled model: {} (PSNR: {:.2f} dB)'.format(best_mn, best_psnr))
visualize_conv_flow_field(
    best_model, dataset, device, timestep_idx=VIS_TIMESTEP,
    title='Online Conv AE ER Scaled ({}): PSNR: {:.2f} dB'.format(best_mn, best_psnr),
    save_path='/kaggle/working/results/conv_er_scaled_flow_field.png', num_windows=NUM_WINDOWS)

---## 3. ER Aggressive

In [ ]:
for model_name in ['base', 'medium', 'large']:
    out_dir = "/kaggle/working/results/conv_er_aggressive_{}/".format(model_name)
    model = create_conv_ae(model_name, input_shape).to(device)
    strategy = STRATEGIES['er_aggressive']()
    metrics = train_online_conv_cl(
        model=model, dataset=dataset, device=device,
        epochs_per_window=EPOCHS_PER_WINDOW, model_name=model_name,
        output_dir=out_dir, strategy=strategy, num_windows=NUM_WINDOWS)
    eval_res = evaluate_full_dataset_conv(model, dataset, device, model_name=model_name, num_windows=NUM_WINDOWS)
    eval_res['training_time_s'] = sum(metrics['time_per_window'])
    all_metrics[(model_name, 'er_aggressive')] = metrics
    all_evals[(model_name, 'er_aggressive')] = eval_res

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for mn, col in [('base', 'tab:blue'), ('medium', 'tab:orange'), ('large', 'tab:green')]:
    m = all_metrics[(mn, 'er_aggressive')]
    axes[0,0].plot(m['window'], m['loss'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
    axes[0,1].plot(m['window'], m['psnr'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
    axes[1,0].plot(m['window'], m['ssim'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
    axes[1,1].plot(m['window'], m['relative_error'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
for ax in axes.flat: ax.legend(); ax.grid(True, alpha=0.3)
axes[0,0].set(xlabel='Window', ylabel='Loss', title='Training Loss')
axes[0,1].set(xlabel='Window', ylabel='PSNR (dB)', title='PSNR')
axes[1,0].set(xlabel='Window', ylabel='SSIM', title='SSIM')
axes[1,1].set(xlabel='Window', ylabel='RE (%)', title='Relative Error')
plt.suptitle('Online Conv AE: ER Aggressive: Per-Window Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/results/conv_er_aggressive_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
best_mn = max(['base', 'medium', 'large'], key=lambda m: all_evals[(m, 'er_aggressive')]['psnr_db'])
best_psnr = all_evals[(best_mn, 'er_aggressive')]['psnr_db']
best_model = create_conv_ae(best_mn, input_shape).to(device)
with torch.no_grad(): best_model(torch.zeros(1, 4, *input_shape).to(device))  # init lazy _fc
best_model.load_state_dict(torch.load(
    '/kaggle/working/results/conv_er_aggressive_{}/{}_final.pth'.format(best_mn, best_mn), map_location=device))
print('Best ER Aggressive model: {} (PSNR: {:.2f} dB)'.format(best_mn, best_psnr))
visualize_conv_flow_field(
    best_model, dataset, device, timestep_idx=VIS_TIMESTEP,
    title='Online Conv AE ER Aggressive ({}): PSNR: {:.2f} dB'.format(best_mn, best_psnr),
    save_path='/kaggle/working/results/conv_er_aggressive_flow_field.png', num_windows=NUM_WINDOWS)

---## 4. Cross-Strategy Comparison

In [ ]:
COMP_DIR = "/kaggle/working/results/comparison_conv_online"
os.makedirs(COMP_DIR, exist_ok=True)

print("=" * 95)
print("  FULL-DATASET EVALUATION: ONLINE CONV2D AE (mesh-level metrics)")
print("=" * 95)
print("{:<12s} {:<16s} {:>10s} {:>10s} {:>10s} {:>10s}".format("Model", "Strategy", "PSNR", "SSIM", "RE(%)", "Time(s)"))
print("-" * 95)
for mn in ['base', 'medium', 'large']:
    r = CONV_OFFLINE_REF[mn]
    print("{:<12s} {:<16s} {:>10.2f} {:>10.4f} {:>10.2f} {:>10s}".format(mn, "Offline(ref)", r["psnr"], r["ssim"], r["re"], "-"))
    for sn in ['naive', 'er_scaled', 'er_aggressive']:
        if (mn, sn) in all_evals:
            ev = all_evals[(mn, sn)]
            print("{:<12s} {:<16s} {:>10.2f} {:>10.4f} {:>10.2f} {:>10.1f}".format(
                mn, sn, ev["psnr_db"], ev["ssim"], ev["relative_error_pct"], ev["training_time_s"]))
    print("")
print("=" * 95)

In [ ]:
# Per-window PSNR: best model per strategy
fig, ax = plt.subplots(figsize=(14, 7))
sm = {'naive': ('Naive', '#F44336', '--'), 'er_scaled': ('ER Scaled', '#FF9800', '-'),
      'er_aggressive': ('ER Aggressive', '#7B1FA2', '-')}
for sn, (lb, co, ls) in sm.items():
    bm = max(['base', 'medium', 'large'], key=lambda m: all_evals[(m, sn)]['psnr_db'])
    m = all_metrics[(bm, sn)]
    ax.plot(m['window'], m['psnr'], ls, color=co, lw=2.5, marker='o', ms=5,
            label='{} ({})'.format(lb, bm))
ax.set(xlabel='Temporal Window', ylabel='PSNR (dB)',
       title='Online Conv AE: Per-Window PSNR: Strategy Comparison')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(os.path.join(COMP_DIR, 'conv_online_psnr_per_window.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Gap to offline: grouped bar chart
fig, ax = plt.subplots(figsize=(14, 7))
x_pos = np.arange(3); width = 0.25
for i, (sn, (lb, co, _)) in enumerate(sm.items()):
    gaps = [CONV_OFFLINE_REF[mn]['psnr'] - all_evals[(mn, sn)]['psnr_db'] for mn in ['base', 'medium', 'large']]
    bars = ax.bar(x_pos + i*width, gaps, width, label=lb, color=co, edgecolor='black', lw=0.5)
    for b, v in zip(bars, gaps):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.2, '{:.1f}'.format(v),
                ha='center', fontsize=9)
ax.set_xticks(x_pos + width); ax.set_xticklabels(['Base', 'Medium', 'Large'])
ax.set(ylabel='PSNR Gap to Offline (dB)', title='Online Conv AE: Gap to Offline (lower is better)')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(COMP_DIR, 'conv_online_gap_to_offline.png'), dpi=300, bbox_inches='tight')
plt.show()

---## 5. Save and Download

In [ ]:
combined = {}
for (mn, sn), ev in all_evals.items():
    combined["{}_{}".format(mn, sn)] = ev
with open(os.path.join(COMP_DIR, 'conv_online_all_results.json'), 'w') as f:
    json.dump(combined, f, indent=2)
print("Results saved. {} experiments.".format(len(combined)))

for (mn, sn), m in all_metrics.items():
    df = pd.DataFrame(m)
    csv_path = "/kaggle/working/results/conv_{}_{}/{}_metrics.csv".format(sn, mn, mn)
    if os.path.exists(os.path.dirname(csv_path)):
        df.to_csv(csv_path, index=False)

import shutil
zip_path = shutil.make_archive('/kaggle/working/conv_online_results', 'zip', '/kaggle/working/results')
print("Archive: {} ({:.1f} MB)".format(zip_path, os.path.getsize(zip_path)/(1024*1024)))